In [1]:
import pandas as pd

In [2]:
!pip install pandas numpy scikit-learn matplotlib seaborn streamlit joblib

In [3]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import (
accuracy_score,
precision_score,
recall_score,
f1_score,
matthews_corrcoef,
roc_auc_score,
confusion_matrix,
classification_report
)

In [4]:
os.makedirs("model", exist_ok = True)

In [5]:
df = pd.read_csv("adult.csv")

print(df.shape)
print(df.columns)
print(df.head())

(6033, 15)
Index(['age', 'workclass', 'fnlwgt', 'education', 'education.num',
       'marital.status', 'occupation', 'relationship', 'race', 'sex',
       'capital.gain', 'capital.loss', 'hours.per.week', 'native.country',
       'target'],
      dtype='object')
   age  workclass  fnlwgt  education  education.num  marital.status  \
0   45          0  207107          7             12               0   
1   22          4  197387         11              9               4   
2   39          2  111268          9             13               4   
3   35          2   98283         14             15               4   
4   40          2  224232          9             13               2   

   occupation  relationship  race  sex  capital.gain  capital.loss  \
0           0             4     1    1             0          2080   
1           4             3     4    1             0             0   
2           0             3     4    0             0             0   
3           9             1   

In [6]:
#Pre-Processing
df.replace('?',np.nan,inplace=True)
df.dropna(inplace=True)
df.shape

(6033, 15)

In [7]:
for col in df.columns:
    if df[col].dtype == 'object':

         le = LabelEncoder()
         df[col] = le.fit_transform(df[col].astype(str))

In [8]:
target = df.columns[-1]

X = df.drop(target, axis = 1)

y = df[target]

print("Target Column:", target)

Target Column: target


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [10]:
test_data = X_test.copy()

test_data[target] = y_test

test_data.to_csv("test_data.csv", index = False)

print("created test data")

created test data


In [11]:
models = {
"Logistic regression" : LogisticRegression(max_iter=1000),

"Decision Tree" : DecisionTreeClassifier(),

"KNN" : KNeighborsClassifier(),

"Naive Bayes" : GaussianNB(),

"SVM" : SVC(probability = True)
}

In [12]:
results = []

for name, model in models.items():
    print(f"\nTraining {name}")

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(y_test, predictions)

    recall = recall_score(y_test, predictions)

    f1 = f1_score(y_test, predictions)

    auc = roc_auc_score(y_test, probabilities)

    mcc = matthews_corrcoef(y_test, predictions)

    results.append([
        name,
        accuracy,
        auc,
        precision,
        recall,
        f1,
        mcc
    ])

    file_name = name.lower().replace(" ","_") + ".pkl"

    joblib.dump(
        model,
        f"model/{file_name}"
    )

    print(f"Saved : {file_name}")

print("All the models are saved")


Training Logistic regression


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Saved : logistic_regression.pkl

Training Decision Tree
Saved : decision_tree.pkl

Training KNN
Saved : knn.pkl

Training Naive Bayes
Saved : naive_bayes.pkl

Training SVM


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Saved : svm.pkl
All the models are saved


In [13]:
results_df = pd.DataFrame(
    results,
    columns = [
        'Model',
        'Accuracy',
        'AUC',
        'Precision',
        'Recall',
        'F1',
        'MCC'
    ]
)

results_df

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic regression,0.802817,0.798244,0.620879,0.400709,0.487069,0.385628
1,Decision Tree,0.803645,0.722758,0.581227,0.570922,0.576029,0.448309
2,KNN,0.749793,0.576860,0.429577,0.216312,0.287736,0.169085
3,Naive Bayes,0.786247,0.814518,0.588235,0.283688,0.382775,0.298623
4,SVM,0.772991,0.603370,1.000000,0.028369,0.055172,0.147939


In [14]:
results_df.sort_values(
    by = "Accuracy",
    ascending = False
)

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
1,Decision Tree,0.803645,0.722758,0.581227,0.570922,0.576029,0.448309
0,Logistic regression,0.802817,0.798244,0.620879,0.400709,0.487069,0.385628
3,Naive Bayes,0.786247,0.814518,0.588235,0.283688,0.382775,0.298623
4,SVM,0.772991,0.603370,1.000000,0.028369,0.055172,0.147939
2,KNN,0.749793,0.576860,0.429577,0.216312,0.287736,0.169085


In [31]:
import os
print(os.listdir("model"))

['svm.pkl', 'logistic_regression.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl', '.ipynb_checkpoints']
